# Phase II · PPO Prompt Selector — Training & Analysis

**Algorithm:** Proximal Policy Optimisation (PPO-Clip)  

---

## Why PPO over A2C?

A2C can suffer from **catastrophic policy updates** — a single bad gradient step can collapse the policy.  
PPO solves this with the **clipped surrogate objective**:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t,\ \text{clip}(r_t(\theta), 1{-}\varepsilon, 1{+}\varepsilon)\hat{A}_t\right)\right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$ is the **importance ratio**.

The clip (ε = 0.2) ensures the new policy never moves too far from the old policy.

**Advantages over A2C:**
- Multiple optimisation epochs per rollout → better sample efficiency  
- Clipping prevents large destabilising updates  
- **GAE** (Generalised Advantage Estimation) reduces variance further

$$\hat{A}_t^{\text{GAE}(\gamma,\lambda)} = \sum_{k=0}^{\infty}(\gamma\lambda)^k \delta_{t+k}$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from mdp_phase2.environment      import MDPCustomerServiceEnv
from mdp_phase2.agents.ppo_agent import PPOAgent
from mdp_phase2.reward           import RewardShaper
from mdp_phase2.train            import train_ppo, evaluate, run_demo_conversation, TrainConfig
from mdp_phase2.strategy_prompts import STRATEGY_NAMES, STRATEGY_LABELS, STRATEGY_COLORS

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = list(STRATEGY_COLORS.values())
print('Imports OK ✓')

## 1 · PPO Hyperparameters

In [ ]:
def make_ppo():
    return PPOAgent(
        state_dim     = MDPCustomerServiceEnv.STATE_DIM,
        num_actions   = MDPCustomerServiceEnv.NUM_ACTIONS,
        gamma         = 0.90,
        lam           = 0.95,   # GAE lambda
        lr            = 3e-4,
        clip_eps      = 0.20,   # PPO clip range
        c_value       = 0.50,
        c_entropy     = 0.01,
        n_epochs      = 4,      # number of gradient passes per rollout
        batch_size    = 32,
        gradient_clip = 0.50,
        rollout_len   = 128,    # steps to collect before updating
        hidden        = (128, 64),
    )

agent_tw = make_ppo()
print('PPO Agent created')
print(f'  clip_eps         = {agent_tw.clip_eps}   (prevents large policy updates)')
print(f'  GAE lambda       = {agent_tw.lam}')
print(f'  n_epochs         = {agent_tw.n_epochs}    (4 passes over each rollout)')
print(f'  rollout_len      = {agent_tw.rollout_len}  (batch size for advantage estimation)')
print(f'  c_entropy        = {agent_tw.c_entropy}   (entropy bonus)')

## 2 · Training — All Three Datasets

In [ ]:
cfg = TrainConfig(
    n_episodes    = 3000,
    eval_every    = 100,
    eval_episodes = 50,
    print_every   = 300,
    seed          = 42,
)

agent_tw = make_ppo()
result_tw = train_ppo(agent_tw, 'twitter', cfg)

agent_rd = make_ppo()
result_rd = train_ppo(agent_rd, 'reddit', cfg)

agent_oa = make_ppo()
result_oa = train_ppo(agent_oa, 'openassistant', cfg)

## 3 · Clipping Analysis

PPO's key contribution is the clipped surrogate.  
We'll visualise the **importance ratio** distribution to confirm the clip is active early  
and tightens as the policy stabilises.

In [ ]:
results    = {'Twitter': result_tw, 'Reddit': result_rd, 'OpenAssistant': result_oa}
agents_map = {'Twitter': agent_tw,  'Reddit': agent_rd,  'OpenAssistant': agent_oa}
ds_colors  = ['#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (ds_name, res), color in zip(axes, results.items(), ds_colors):
    eval_eps = [ev.episode            for ev in res.eval_snapshots]
    sr       = [ev.success_rate * 100 for ev in res.eval_snapshots]
    smooth   = res.smooth_rewards(50)

    # plot both reward and success rate on twin axes
    ax2 = ax.twinx()
    offset = len(res.episode_rewards) - len(smooth)
    ax.plot(res.episode_rewards, alpha=0.1, color=color, linewidth=0.4)
    ax.plot(range(offset, offset+len(smooth)), smooth, color=color, linewidth=2, label='Reward')
    ax2.plot(eval_eps, sr, color='#D62728', linewidth=2, linestyle='--', 
             marker='o', ms=3, label='Success %')

    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward', color=color)
    ax2.set_ylabel('Success Rate (%)', color='#D62728')
    ax2.set_ylim(0, 100)
    ax.set_title(f'PPO — {ds_name}', fontweight='bold')

    # add legend for both axes
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=8)

plt.suptitle('PPO Training: Reward & Success Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ppo_training_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Final Strategy Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (ds_name, res), color in zip(axes, results.items(), ds_colors):
    final_ev = res.eval_snapshots[-1]
    vals = [final_ev.strategy_dist.get(n, 0) * 100 for n in STRATEGY_NAMES]
    bars = ax.bar(STRATEGY_LABELS, vals, color=COLORS, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.1f}%', ha='center', fontsize=8)
    ax.set_title(f'Strategy Mix — {ds_name}', fontweight='bold')
    ax.set_ylabel('Usage (%)')
    ax.set_ylim(0, max(vals) + 10)
    ax.set_xticklabels(STRATEGY_LABELS, rotation=25, ha='right')

plt.suptitle('PPO — Final Strategy Distributions Across Datasets', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ppo_strategy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · Demo Conversation — PPO Agent

In [ ]:
# show a reddit conversation (longer, more nuanced)
demo = run_demo_conversation(agent_rd, dataset='reddit', verbose=True)

## 6 · Summary

In [ ]:
print('=' * 60)
print('  PPO PHASE II — TRAINING SUMMARY')
print('=' * 60)
for ds_name, res in results.items():
    ev = res.eval_snapshots[-1]
    print(f'\n  {ds_name}')
    print(f'    Resolution rate : {ev.success_rate*100:.1f}%')
    print(f'    Escalation rate : {ev.escalation_rate*100:.1f}%')
    print(f'    Avg reward      : {ev.avg_reward:+.3f}')
    print(f'    Avg turns       : {ev.avg_turns:.1f}')
    print(f'    Training time   : {res.train_time_s:.1f}s')
print('\n  Next: comparison_phase2.ipynb for DQN vs A2C vs PPO vs Phase I')
print('=' * 60)